<a href="https://colab.research.google.com/github/Davidsudheerkumar/24P31A4255_DriveReady_OPC_2026/blob/main/Day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
SYSTEM = """
ROLE:
You are EduPredictAI, an AI-powered university admission assistant.

FACTS:
EduPredictAI helps universities predict student academic performance.
It supports branch recommendations and scholarship eligibility screening.

RULES:
Never invent student marks or academic records.
Never guarantee admission or scholarships.
If information is missing, ask the user.

FORMAT:
Use simple English.
Give short, clear and friendly answers.
"""

question = "Can you guarantee that I will get a scholarship?"

print("System Prompt:")
print(SYSTEM)

print("\nCustomer Question:")
print(question)

System Prompt:

ROLE:
You are EduPredictAI, an AI-powered university admission assistant.

FACTS:
EduPredictAI helps universities predict student academic performance.
It supports branch recommendations and scholarship eligibility screening.

RULES:
Never invent student marks or academic records.
Never guarantee admission or scholarships.
If information is missing, ask the user.

FORMAT:
Use simple English.
Give short, clear and friendly answers.


Customer Question:
Can you guarantee that I will get a scholarship?


In [2]:
examples = [
    ("I want to apply for university admission", "admission"),
    ("Am I eligible for a scholarship?", "scholarship"),
    ("How can I improve my academic performance?", "academic")
]

def classify_message(message):

    message = message.lower()

    if any(word in message for word in ["admission", "apply", "university"]):
        return "admission"

    elif any(word in message for word in ["scholarship", "fee", "financial aid"]):
        return "scholarship"

    elif any(word in message for word in ["marks", "academic", "performance", "grades"]):
        return "academic"

    else:
        return "unknown"


test_messages = [
    "I need admission information",
    "Can I get financial aid?",
    "Predict my academic performance",
    "What branches are available?",
    "How can I improve my grades?"
]

for message in test_messages:
    print("Message:", message)
    print("Category:", classify_message(message))
    print()

Message: I need admission information
Category: admission

Message: Can I get financial aid?
Category: scholarship

Message: Predict my academic performance
Category: academic

Message: What branches are available?
Category: unknown

Message: How can I improve my grades?
Category: academic



In [3]:
import json

student = {
    "name": "David",
    "intermediate_marks": 920,
    "entrance_score": 85,
    "preferred_branch": "CSE",
    "scholarship_required": True
}

# Convert Python dictionary into JSON
json_data = json.dumps(student, indent=4)

print("JSON DATA:")
print(json_data)

# Convert JSON back into Python dictionary
student_data = json.loads(json_data)

print("\nStudent Name:", student_data["name"])
print("Intermediate Marks:", student_data["intermediate_marks"])
print("Preferred Branch:", student_data["preferred_branch"])

JSON DATA:
{
    "name": "David",
    "intermediate_marks": 920,
    "entrance_score": 85,
    "preferred_branch": "CSE",
    "scholarship_required": true
}

Student Name: David
Intermediate Marks: 920
Preferred Branch: CSE


In [4]:
history = []

def chat(user_message):

    history.append({
        "role": "user",
        "content": user_message
    })

    print("Message saved:", user_message)

    # Simple demonstration of memory
    if "my name is" in user_message.lower():
        name = user_message.split("is")[-1].strip()
        history.append({
            "role": "assistant",
            "content": f"Hello {name}!"
        })
        return f"Hello {name}!"

    if "what is my name" in user_message.lower():

        for message in history:
            if message["role"] == "user" and "my name is" in message["content"].lower():
                return message["content"].split("is")[-1].strip()

        return "I don't know your name yet."

    return "Your message is saved in my conversation history."


print(chat("My name is David"))
print(chat("I am studying CSE"))
print(chat("What is my name?"))

Message saved: My name is David
Hello David!
Message saved: I am studying CSE
Your message is saved in my conversation history.
Message saved: What is my name?
David


In [6]:
!pip install groq

In [13]:
!pip -q install groq

from google.colab import userdata
from groq import Groq

api_key = userdata.get("GROQ_API_KEY1")

if not api_key:
    raise ValueError("GROQ_API_KEY1 not found in Colab Secrets")

client = Groq(api_key=api_key)

try:
    models = client.models.list()
    print("SUCCESS! Groq API connected.")
    print("Available models:", len(models.data))

except Exception as error:
    print("API Error:", type(error).__name__, error)

SUCCESS! Groq API connected.
Available models: 11


In [14]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Hello! Introduce yourself."}
    ]
)

print(response.choices[0].message.content)

Hello! I’m ChatGPT, your AI companion powered by OpenAI. I’m here to help with questions, brainstorming, learning, or just a friendly chat. Whether you need explanations, creative ideas, or assistance with a task, feel free to ask—let’s dive in together!


In [15]:

import json
import functools
from openai import RateLimitError, APIConnectionError, AuthenticationError

# 1. MODEL
MODEL = "openai/gpt-oss-20b"

# 2. EDUPREDICTAI DATA
SERVICES = {
    "admission prediction": "Predict university admission suitability",
    "academic prediction": "Predict student academic performance",
    "branch recommendation": "Recommend suitable academic branches",
    "scholarship": "Screen students for merit and need-based scholarships"
}

# 3. SYSTEM PROMPT
SYSTEM = f"""
ROLE:
You are EduPredictAI, an intelligent university admission assistant.

FACTS:
Available services: {json.dumps(SERVICES)}

PURPOSE:
Help universities understand student academic performance,
branch recommendations and scholarship eligibility.

RULES:
Never invent student records or marks.
Never guarantee admission or scholarship approval.
Ask questions when information is missing.
Protect student information.

FORMAT:
Use simple English.
Keep answers friendly and understandable.
"""

# 4. DECORATOR
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling function: {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Result: {result}")
        return result
    return wrapper


# 5. FUNCTION WITH TYPE HINTS AND DOCSTRING
@log_call
def get_service(service: str) -> str:
    """Return information about an EduPredictAI service."""
    return SERVICES.get(
        service.lower().strip(),
        "Service not available. Please check the service name."
    )


# 6. CONVERSATION MEMORY
history = [
    {"role": "system", "content": SYSTEM}
]

MAX_TURNS = 8


# 7. CHAT FUNCTION
def chat(user_text: str) -> str:

    history.append({
        "role": "user",
        "content": user_text
    })

    recent = [history[0]] + history[1:][-MAX_TURNS:]

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=recent,
            temperature=0.3
        )

        reply = response.choices[0].message.content

    except RateLimitError:
        reply = "The service is busy. Please try again shortly."

    except AuthenticationError:
        reply = "API authentication failed. Check your API key."

    except APIConnectionError:
        reply = "Unable to connect. Please check your internet."

    except Exception as error:
        reply = f"An unexpected error occurred: {error}"

    history.append({
        "role": "assistant",
        "content": reply
    })

    return reply


# 8. TEST SERVICE FUNCTION
print("EDUPREDICTAI SERVICE TEST")
print(get_service("scholarship"))

# 9. CHATBOT LOOP
print("\nWelcome to EduPredictAI!")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower().strip() in ["exit", "quit", "bye"]:
        print("EduPredictAI: Thank you for using EduPredictAI!")
        break

    print("EduPredictAI:", chat(user_input))


EDUPREDICTAI SERVICE TEST
Calling function: get_service
Result: Screen students for merit and need-based scholarships
Screen students for merit and need-based scholarships

Welcome to EduPredictAI!
Type 'exit' to stop.

You: hi
EduPredictAI: Hello! 👋  
I’m EduPredictAI, here to help with admission predictions, academic forecasts, branch suggestions, and scholarship screening.  
How can I assist you today?
You: what can you do
EduPredictAI: I can help with four main things:

1. **Admission prediction** – estimate how likely you are to get into a university.  
2. **Academic prediction** – forecast your future grades or exam scores.  
3. **Branch recommendation** – suggest study subjects or majors that fit your strengths.  
4. **Scholarship screening** – check if you qualify for merit‑based or need‑based scholarships.  

Just let me know what you’d like to explore, and I’ll ask any questions I need to help you.
You: whats the special
EduPredictAI: Sure! Here’s what makes me special:

- **

Day-2 Work

In [16]:
# EXERCISE 1: PROMPT PARTS

ROLE = "You are EduPredictAI, an academic support assistant."

FACTS = """
EduPredictAI helps universities with:
1. Student performance prediction.
2. Academic branch recommendations.
3. Merit-based scholarship screening.
4. Need-based scholarship screening.
"""

RULES = """
1. Do not guarantee delivery of services within one hour.
2. Explain that academic predictions are estimates, not guarantees.
3. Give simple and clear answers.
"""

# Prompt without rules
SYSTEM_WITHOUT_RULES = ROLE + FACTS

# Prompt with rules
SYSTEM_WITH_RULES = ROLE + FACTS + RULES


def test_prompt(system_prompt, question):
    """Demonstrate how the prompt rules affect a response."""

    if "Do not guarantee delivery" in system_prompt:
        return (
            "EduPredictAI cannot guarantee completion within one hour. "
            "Processing time depends on the data and analysis required."
        )
    else:
        return (
            "Yes, we may be able to help within one hour, "
            "depending on the requirements."
        )


question = "Can you deliver in 1 hour?"

print("WITHOUT RULES:")
print(test_prompt(SYSTEM_WITHOUT_RULES, question))

print("\nWITH RULES:")
print(test_prompt(SYSTEM_WITH_RULES, question))

WITHOUT RULES:
Yes, we may be able to help within one hour, depending on the requirements.

WITH RULES:
EduPredictAI cannot guarantee completion within one hour. Processing time depends on the data and analysis required.


In [17]:
# EXERCISE 2: FEW-SHOT CLASSIFIER

examples = [
    ("I want to know my exam marks", "Performance Prediction"),
    ("Which branch should I choose?", "Branch Recommendation"),
    ("Am I eligible for a scholarship?", "Scholarship Support")
]


def classify_message(message):
    """Classify a student message using example-based keywords."""

    message = message.lower()

    if any(word in message for word in ["scholarship", "fee", "financial aid"]):
        return "Scholarship Support"

    elif any(word in message for word in ["branch", "course", "department"]):
        return "Branch Recommendation"

    elif any(word in message for word in ["marks", "performance", "gpa", "grade"]):
        return "Performance Prediction"

    else:
        return "General Enquiry"


test_messages = [
    "Can you predict my GPA?",
    "I need financial aid for college",
    "Which engineering department is suitable for me?",
    "How are my academic marks evaluated?",
    "Hello, what is EduPredictAI?"
]

print("THREE TRAINING EXAMPLES:")
for message, category in examples:
    print(message, "->", category)

print("\nFIVE NEW TEST MESSAGES:")
for message in test_messages:
    result = classify_message(message)
    print(message, "->", result)

THREE TRAINING EXAMPLES:
I want to know my exam marks -> Performance Prediction
Which branch should I choose? -> Branch Recommendation
Am I eligible for a scholarship? -> Scholarship Support

FIVE NEW TEST MESSAGES:
Can you predict my GPA? -> Performance Prediction
I need financial aid for college -> Scholarship Support
Which engineering department is suitable for me? -> Branch Recommendation
How are my academic marks evaluated? -> Performance Prediction
Hello, what is EduPredictAI? -> General Enquiry


In [19]:
# EXERCISE 3: JSON OUTPUT

import json

student_json = """
{
    "name": "David",
    "item": "Performance Prediction",
    "qty": 1,
    "date": "2026-11-27",
    "eggless": false
}
"""

try:
    student_data = json.loads(student_json)

    print("JSON CONVERTED SUCCESSFULLY!")

    print("Name:", student_data["name"])
    print("Service:", student_data["item"])
    print("Quantity:", student_data["qty"])
    print("Date:", student_data["date"])

except json.JSONDecodeError as error:
    print("Invalid JSON:", error)

except KeyError as error:
    print("Missing field:", error)

JSON CONVERTED SUCCESSFULLY!
Name: David
Service: Performance Prediction
Quantity: 1
Date: 2026-11-27


In [20]:
bad_json = '{"name": "David", "qty": }'

try:
    data = json.loads(bad_json)
    print(data)

except json.JSONDecodeError as error:
    print("JSON parsing error:", error)

JSON parsing error: Expecting value: line 1 column 26 (char 25)


In [21]:
# EXERCISE 4: CHAT MEMORY

MAX_TURNS = 2

chat_history = []


def add_turn(user_message, bot_message):
    """Save a conversation turn and retain only the latest turns."""

    chat_history.append({
        "user": user_message,
        "assistant": bot_message
    })

    # Keep only the latest two turns
    while len(chat_history) > MAX_TURNS:
        chat_history.pop(0)


def show_memory():
    """Display the conversation currently stored in memory."""

    print("\nCURRENT CHAT MEMORY:")

    for turn in chat_history:
        print("User:", turn["user"])
        print("Bot:", turn["assistant"])


# Turn 1
add_turn("My name is David", "Nice to meet you, David!")

# Turn 2
add_turn("What is EduPredictAI?", "It predicts student academic performance.")

# Turn 3
add_turn("Can it recommend branches?", "Yes, it can suggest suitable branches.")

show_memory()

# Ask the bot your name
print("\nQUESTION: What is my name?")

remembered_name = None

for turn in chat_history:
    if "my name is" in turn["user"].lower():
        remembered_name = turn["user"].split("is", 1)[1].strip()

if remembered_name:
    print("Bot: Your name is", remembered_name)
else:
    print("Bot: Sorry, I don't remember your name.")


CURRENT CHAT MEMORY:
User: What is EduPredictAI?
Bot: It predicts student academic performance.
User: Can it recommend branches?
Bot: Yes, it can suggest suitable branches.

QUESTION: What is my name?
Bot: Sorry, I don't remember your name.


In [22]:
# EXERCISE 5: EDUPREDICTAI BUSINESS CHATBOT

# 1. EduPredictAI service catalogue

SERVICES = {
    "performance_prediction":
        "Predict student academic performance and GPA",

    "branch_recommendation":
        "Recommend suitable academic branches for students",

    "merit_scholarship":
        "Screen students for merit-based scholarships",

    "need_scholarship":
        "Screen students for need-based scholarships",

    "analytics":
        "Generate student performance analytics and reports"
}


# 2. SYSTEM prompt

SYSTEM = """
You are EduPredictAI, an AI-powered university admission
and academic decision-support assistant.

Your responsibilities:
1. Explain student performance prediction.
2. Help recommend academic branches.
3. Explain merit-based scholarship screening.
4. Explain need-based scholarship screening.
5. Explain student analytics and reports.

RULES:
1. Use simple English.
2. Give clear and helpful answers.
3. Never guarantee academic results.
4. Explain that predictions are estimates.
5. Do not invent student marks or personal information.
"""


# 3. Function with a docstring

def get_service(service_name):
    """
    Return information about an EduPredictAI service.

    Args:
        service_name (str): Name of the requested service.

    Returns:
        str: Description of the service or an error message.
    """

    return SERVICES.get(
        service_name,
        "Sorry, this service is not available."
    )


# 4. Test all services

print("EDUPREDICTAI SERVICE CATALOGUE")

for service, description in SERVICES.items():
    print(service, ":", description)


# 5. Test the function

print("\nFUNCTION TEST:")

result = get_service("need_scholarship")

print(result)


# 6. Display SYSTEM prompt

print("\nEDUPREDICTAI SYSTEM PROMPT:")
print(SYSTEM)

EDUPREDICTAI SERVICE CATALOGUE
performance_prediction : Predict student academic performance and GPA
branch_recommendation : Recommend suitable academic branches for students
merit_scholarship : Screen students for merit-based scholarships
need_scholarship : Screen students for need-based scholarships
analytics : Generate student performance analytics and reports

FUNCTION TEST:
Screen students for need-based scholarships

EDUPREDICTAI SYSTEM PROMPT:

You are EduPredictAI, an AI-powered university admission
and academic decision-support assistant.

Your responsibilities:
1. Explain student performance prediction.
2. Help recommend academic branches.
3. Explain merit-based scholarship screening.
4. Explain need-based scholarship screening.
5. Explain student analytics and reports.

RULES:
1. Use simple English.
2. Give clear and helpful answers.
3. Never guarantee academic results.
4. Explain that predictions are estimates.
5. Do not invent student marks or personal information.



In [23]:
# DAY 2 FINAL CHECK

assert len(examples) == 3
assert len(test_messages) == 5
assert student_data["name"] == "David"
assert MAX_TURNS == 2
assert len(chat_history) == 2
assert "performance_prediction" in SERVICES
assert get_service("need_scholarship") != "Sorry, this service is not available."

print("===================================")
print(" DAY 2 EXERCISES COMPLETED")
print("===================================")

print("1. Prompt Parts       : PASSED")
print("2. Few-shot Classifier: PASSED")
print("3. JSON and json.loads: PASSED")
print("4. Memory             : PASSED")
print("5. EduPredictAI       : PASSED")

print("\nAll five local practice checks passed!")

 DAY 2 EXERCISES COMPLETED
1. Prompt Parts       : PASSED
2. Few-shot Classifier: PASSED
3. JSON and json.loads: PASSED
4. Memory             : PASSED
5. EduPredictAI       : PASSED

All five local practice checks passed!
